# BreastDivider dataset - wstepne rozpoznanie

Notebook podsumowuje Etap 1 pracowni problemowej: lokalne pobranie malej probki BreastDivider, sprawdzenie struktury plikow, podstawowe EDA plikow NIfTI oraz wizualizacje obrazow MRI z maskami lewej i prawej piersi.

Celem tego notebooka nie jest trenowanie modelu, tylko pokazanie, ze dataset zostal technicznie rozpoznany i ze mozna go wykorzystac do przygotowania pipeline'u slice-level dla dalszej pracy nad embeddingami i virtual DCE.

## 1. Kontekst

BreastDivider jest publicznym datasetem MRI piersi z maskami segmentacji lewej i prawej piersi. W kontekscie pracy _Breast MRI Embedding Models for Virtual Dynamic Contrast-Enhanced Image Synthesis_ dataset jest istotny przede wszystkim jako zrodlo danych i masek anatomicznych.

Maski left/right moga zostac wykorzystane do:

- wyboru przekrojow zawierajacych tkanke piersi,
- ograniczenia analizy do obszaru piersi,
- cropowania lub maskowania obrazu,
- odrzucania pustych lub malo informacyjnych slice'ow,
- pozniejszej ewaluacji rekonstrukcji w masce piersi.

In [ ]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data' / 'BreastDividerDataset'
OUT_DIR = ROOT / 'outputs' / 'breastdivider_eda'
VIS_DIR = OUT_DIR / 'visualizations'

print('ROOT:', ROOT)
print('DATA_DIR exists:', DATA_DIR.exists(), DATA_DIR)
print('OUT_DIR exists:', OUT_DIR.exists(), OUT_DIR)
print('VIS_DIR exists:', VIS_DIR.exists(), VIS_DIR)

## 2. Pliki wynikowe wygenerowane w Etapie 1

W Etapie 1 wykorzystano trzy skrypty:

- `inspect_breastdivider_structure.py` - inwentaryzacja pobranych plikow,
- `eda_nifti_summary.py` - podstawowe statystyki wolumenow NIfTI,
- `visualize_breastdivider_sample.py` - wizualizacja MRI + maska left/right.

Ponizsza komorka wczytuje wygenerowane pliki CSV.

In [ ]:
file_inventory_path = OUT_DIR / 'file_inventory.csv'
nifti_summary_path = OUT_DIR / 'nifti_summary.csv'
visual_summary_path = OUT_DIR / 'sample_visual_summary.csv'
metadata_summary_path = OUT_DIR / 'metadata_summary.txt'

file_inventory = pd.read_csv(file_inventory_path)
nifti_summary = pd.read_csv(nifti_summary_path)
visual_summary = pd.read_csv(visual_summary_path)

print('file_inventory rows:', len(file_inventory))
print('nifti_summary rows:', len(nifti_summary))
print('visual_summary rows:', len(visual_summary))

if metadata_summary_path.exists():
    print('\nmetadata_summary.txt:')
    print(metadata_summary_path.read_text(encoding='utf-8'))

## 3. Co zostalo pobrane?

Pobrano metadane datasetu oraz mala probke obrazow i masek. W tej probce celem jest sprawdzenie poprawnosci formatu, parowania obraz-maska i podstawowej uzytecznosci masek, a nie analiza statystyczna calego datasetu.

In [ ]:
inventory_no_cache = file_inventory[~file_inventory['relative_path'].str.startswith('.cache/')].copy()

display(inventory_no_cache.groupby('kind').agg(files=('relative_path', 'count'), total_mb=('size_bytes', lambda s: round(s.sum() / 1024**2, 2))).reset_index())

display(inventory_no_cache[['relative_path', 'kind', 'size_bytes']])

In [ ]:
image_re = re.compile(r'imagesTr_batch\d+/BreastDivider_(\d{5})_0000\.nii\.gz$')
mask_re = re.compile(r'labelsTr_batch\d+/BreastDivider_(\d{5})\.nii\.gz$')

image_ids = set()
mask_ids = set()

for rel_path in inventory_no_cache['relative_path']:
    image_match = image_re.search(rel_path)
    mask_match = mask_re.search(rel_path)
    if image_match:
        image_ids.add(image_match.group(1))
    if mask_match:
        mask_ids.add(mask_match.group(1))

paired_ids = sorted(image_ids & mask_ids)
missing_images = sorted(mask_ids - image_ids)
missing_masks = sorted(image_ids - mask_ids)

print('Images:', len(image_ids), sorted(image_ids))
print('Masks:', len(mask_ids), sorted(mask_ids))
print('Complete image-mask pairs:', len(paired_ids), paired_ids)
print('Masks without image:', missing_images)
print('Images without mask:', missing_masks)

## 4. Podstawowe statystyki NIfTI

Ta sekcja pokazuje, czy pliki `.nii.gz` otwieraja sie poprawnie oraz jakie maja rozmiary, spacing, typ danych i zakresy intensywnosci. Jest to wazne, poniewaz preprocessing slice-level nie moze zakladac jednego stalego ksztaltu lub jednego zakresu intensywnosci.

In [ ]:
display(nifti_summary)

errors = nifti_summary[nifti_summary.get('error', pd.Series([''] * len(nifti_summary))).notna() & (nifti_summary.get('error', '') != '')]
print('Rows with loading errors:', len(errors))
if len(errors):
    display(errors[['relative_path', 'error']])

In [ ]:
summary_cols = ['relative_path', 'kind', 'shape', 'zooms', 'dtype', 'min', 'p01', 'mean', 'p99', 'max']
display(nifti_summary[summary_cols])

print('Shapes:')
display(nifti_summary.groupby(['kind', 'shape']).size().reset_index(name='count'))

print('Spacing / zooms:')
display(nifti_summary.groupby(['kind', 'zooms']).size().reset_index(name='count'))

print('Dtypes:')
display(nifti_summary.groupby(['kind', 'dtype']).size().reset_index(name='count'))

## 5. Zgodnosc obrazow i masek

Dla dalszego pipeline'u kluczowe jest, aby obraz MRI i odpowiadajaca mu maska mialy ten sam rozmiar. W tej probce sparowane przypadki maja zgodne `image_shape` i `mask_shape`.

In [ ]:
visual_summary['shape_match'] = visual_summary['image_shape'] == visual_summary['mask_shape']
display(visual_summary[['case_id', 'image_shape', 'mask_shape', 'shape_match', 'selected_axis', 'selected_slice', 'foreground_slices_axis0', 'foreground_slices_axis1', 'foreground_slices_axis2', 'label_counts']])

print('All visualized pairs have matching shape:', bool(visual_summary['shape_match'].all()))

## 6. Wizualizacje MRI + maska left/right

Ponizsze obrazy pokazuja wybrane przekroje wolumenow MRI oraz maski lewej i prawej piersi. To najprostszy wizualny test, czy obraz i maska sa poprawnie sparowane oraz czy maska obejmuje anatomicznie sensowny obszar.

In [ ]:
png_paths = sorted(VIS_DIR.glob('*.png'))
print('PNG visualizations:', len(png_paths))

for path in png_paths[:6]:
    display(Markdown(f'### {path.name}'))
    display(Image(filename=str(path)))

## 7. Wnioski z Etapu 1

Na podstawie pobranej probki mozna sformulowac nastepujace wnioski:

1. Udalo sie pobrac metadane BreastDivider oraz mala probke obrazow i masek w formacie `.nii.gz`. Oznacza to, ze dataset jest technicznie dostepny lokalnie i mozna go analizowac standardowymi narzedziami do obrazow medycznych.
2. Pliki NIfTI otwieraja sie poprawnie, a dla sparowanych przypadkow obraz i maska maja zgodne wymiary. To potwierdza, ze podstawowy schemat parowania `image -> mask` dziala i moze byc wykorzystany w dalszym preprocessingu.
3. Maski zawieraja oczekiwane etykiety `0`, `1`, `2`, odpowiadajace tlu oraz lewej i prawej piersi. Taki format etykiet jest prosty do wykorzystania w skryptach: mozna latwo tworzyc maske calej piersi albo osobne maski strony lewej i prawej.
4. Wizualizacje overlay potwierdzaja, ze maski obejmuja anatomicznie sensowne regiony piersi. Jest to wazne, poniewaz w pracy planowany model embeddingowy ma operowac na przekrojach 2D, a maski moga pomoc wybrac tylko te slice'y, ktore zawieraja rzeczywista tkanke piersi.
5. Dataset wykazuje duza zmiennosc shape, spacing, dtype i zakresow intensywnosci. W probce wystepuja m.in. rozmiary `448x448x120`, `512x512x116`, `512x512x192` oraz `480x160x480`, a spacing i typ danych roznia sie miedzy przypadkami. To oznacza, ze preprocessing nie moze zakladac jednego stalego rozmiaru danych.
6. Dalszy pipeline powinien uwzgledniac kontrolowana normalizacje intensywnosci, wybor osi przekrojow, crop lub resize oraz odrzucanie slice'ow z mala powierzchnia maski piersi. Bez tych krokow model moglby uczyc sie roznic technicznych miedzy akwizycjami zamiast cech istotnych dla rekonstrukcji obrazu.
7. BreastDivider nadaje sie do przygotowania pipeline'u slice-level: wyboru przekrojow z tkanka piersi, maskowania/cropowania oraz odrzucania pustych przekrojow. Jest to zgodne z pierwsza faza planowanej pracy, czyli uczeniem modelu embeddingowego na pojedynczych przekrojach 2D MRI piersi.
8. Na podstawie tej probki mozna stwierdzic, ze BreastDivider jest szczegolnie uzyteczny jako dataset anatomiczny i preprocessingowy. Nie jest to jeszcze potwierdzenie, ze dataset samodzielnie wystarczy do trenowania virtual DCE, poniewaz do tego potrzebne sa jednoznaczne pary pre-contrast/post-contrast.
9. Probka techniczna nie wystarcza do wnioskow statystycznych ani trenowania modelu. Jej wartosc polega na walidacji formatu danych, sprawdzeniu parowania obrazow i masek oraz przygotowaniu podstaw do wiekszej analizy datasetu.

Wazna uwaga techniczna: w obecnej probce pobrano maski dla przypadkow `00001-00010`, ale obraz `00001` nie znajduje sie w lokalnej probce. Dlatego wizualizacje obejmuja 9 kompletnych par `00002-00010`. Nie blokuje to Etapu 1, poniewaz celem bylo sprawdzenie technicznej uzytecznosci danych, ale warto odnotowac te roznice w opisie probki.